# 🌽 Futures Markets & Central Counterparties — a field guide for people who'd rather not read a textbook

Everything below is a **real mechanic of real futures markets**, explained like
you're a smart friend who was on their phone during the lecture. Featuring: corn
that never arrives, a man who accidentally bought a herd of cattle, and a hedge
fund that did calculus at the market and lost four billion dollars.

**How to run:** top to bottom. Every cell is Python 3 + `numpy` + `matplotlib` —
runs anywhere, including a plain Google Colab runtime (*Runtime ▸ Run all*). One
optional section (§ 5½) lights up [`nablatensor`](https://github.com/nablatensor-dev/nablatensor)
for an adjoint-AD detour: **on Colab its first run builds the project itself**;
locally it uses whatever install you have; with neither, it simply skips itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

rng = np.random.default_rng(2)
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
})

def rule(title):
    print("\n" + title + "\n" + "-" * len(title))

print("kit ready — numpy", np.__version__)

## 1 · How a futures contract is born (it takes two)

Picture a trader in New York who wants **5,000 bushels of corn** delivered in
September. They call a broker; the broker tells a floor trader to **buy one
September corn contract** (each is exactly 5,000 bushels — the exchange does not
negotiate). At roughly the same moment a trader in Kansas wants to *sell* 5,000
bushels of September corn, and their broker says **sell one contract**. A
computer (or, in the old days, two humans yelling in a pit) matches them at a
price — say **600 cents per bushel**. Neither trader knows or cares who the other
one is. It's a dating app where the only compatibility question is "corn, September?"

- **Long** = you agreed to buy. **Short** = you agreed to sell.
- The **futures price** moves on plain supply and demand: more sellers than
  buyers → price drops until new buyers appear; more buyers → price rises until
  new sellers appear.
- Your profit per contract = *(price change) × (contract size) × (+1 if long, −1 if short)*.
  It's a bet on a straight line.

In [ ]:
F0, size = 600.0, 5000                 # cents/bushel, bushels per contract
FT = np.linspace(520, 680, 200)
long_pnl  = size * (FT - F0) / 100     # dollars
short_pnl = size * (F0 - FT) / 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(FT, long_pnl,  lw=2, label="Long 1 contract  (you agreed to BUY)")
ax.plot(FT, short_pnl, lw=2, label="Short 1 contract  (you agreed to SELL)")
ax.axhline(0, color="k", lw=.8)
ax.axvline(F0, color="gray", ls="--", lw=.8)
ax.annotate("price you locked in\n600 ¢/bu", (F0, 0), (F0 + 8, -1500),
            arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_xlabel("Corn price later (¢/bushel)")
ax.set_ylabel("Your profit ($)")
ax.set_title("A futures contract is a bet on a straight line")
ax.legend(); fig.tight_layout(); plt.show()

print(f"A +10 ¢/bu move → long makes ${size*10/100:,.0f}, short loses exactly the same.")

## 2 · Closing out: how 98% of traders never touch a single bushel

Almost no futures contract ends in delivery. You get out by doing the **exact
opposite trade** before the delivery month. Bought a September contract on June 5?
Sell a September contract on July 20 and you're flat — your entire profit or loss
is just the change in the futures price between those two dates. Shorted one on
June 5? Buy one back on August 25. Done.

Delivery is so rare that people forget it can happen at all.

> **Field note — the "$0 → one herd of cattle" speedrun.**
> A new hire, fresh from outside finance, inherits a client who every month goes
> **long one live-cattle contract** (40,000 lb of cow) purely to hedge, and
> always closes it on the last trading day. The instruction said *"close out the
> position."* The new hire saw *"long,"* thought *"buy,"* and bought **another
> one**. The firm is now long **two** herds and trading has stopped. Being long,
> there is nothing to do but wait for some short to point a finger. The finger
> arrives: 40,000 lb of live cattle, collect **Tuesday, 2,000 miles away**. The
> delivering short buys the cattle at the local Tuesday auction and hands them
> over on the spot — and they can't be re-sold until *next* Tuesday's auction. So
> week one of this person's finance career is spent arranging room and board for
> a herd of cattle. Welcome to the industry.

## 3 · The contract spec: the exchange is a lovable control freak

Before a contract can trade, the exchange nails down **every** detail so the two
sides never end up in an argument:

- **The asset and its grade.** Physical stuff varies, so the exchange picks
  acceptable grades. Orange juice: US Grade A, Brix ≥ 62.5°. Corn: standard is
  "No. 2 Yellow"; No. 1 Yellow delivers for **+1.5 ¢/bu**, No. 3 Yellow for
  **−2 to −4 ¢/bu**. The Treasury-bond contract accepts *any* US Treasury bond
  with **15–25 years** to maturity, with a formula to adjust the price for the
  actual coupon and maturity delivered. A Japanese yen, mercifully, needs no grade.
- **Contract size.** Too big and small hedgers are locked out; too small and the
  per-contract costs eat you. Farm products run ~$10k–$20k of goods; a T-bond
  contract delivers **$100,000 face**. "Mini" contracts exist to let smaller
  traders in (Mini Nasdaq-100 = 20× the index vs the regular 100×).
- **Where delivery happens.** Matters when freight is pricey. OJ delivers to
  licensed warehouses in Florida, New Jersey or Delaware; locations far from the
  source often fetch a *higher* price for the short.
- **Delivery months.** The contract is *named* after its delivery month. Corn
  trades March, May, July, September, December. The exchange sets the first and
  the last trading day (trading usually stops a few days before delivery can begin).
- **Price quotes.** Crude oil in dollars and cents. Treasuries in dollars and
  **thirty-seconds** of a dollar, because tradition is undefeated.
- **Price limits.** The daily move is capped. Hit the ceiling and the contract is
  **limit up**; hit the floor and it's **limit down**; trading usually stops for
  the day. Meant to cool speculative stampedes — but when the real price is
  genuinely running, the limit is just a wall you stare at. Whether limits help
  at all is a genuinely unsettled argument.
- **Position limits.** The most contracts a speculator may hold, so no single
  whale can move the pond.

In [ ]:
limit, true_jump = 3.0, 22.0           # daily cap vs where the market wants to gap
days = np.arange(0, 12)
reported = [100.0]
for _ in days[1:]:
    step = np.clip((100 + true_jump) - reported[-1], -limit, limit)
    reported.append(reported[-1] + step)
reported = np.array(reported)
locked = np.abs(np.diff(reported, prepend=reported[0])) >= limit - 1e-9

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.axhline(100 + true_jump, color="tab:red", ls="--", lw=1,
           label="where the price actually wants to be")
ax.step(days, reported, where="mid", lw=2, label="what the exchange lets it print")
ax.scatter(days[locked], reported[locked], color="tab:red", zorder=5,
           label="LIMIT UP — trading halts for the day")
ax.set_xlabel("Trading day"); ax.set_ylabel("Futures price")
ax.set_title("Limit up: the price takes the stairs, not the elevator")
ax.legend(); fig.tight_layout(); plt.show()

print(f"{locked.sum()} of {len(days)} days were limit-up locked before the price got where it wanted.")

## 4 · Convergence: futures and spot always end up together

As the delivery period approaches, the **futures price converges to the spot
price**. During the delivery period they're basically equal. Why? Free money
otherwise:

- **Futures above spot in the delivery period?** Short the future, buy the asset
  now, deliver it — pocket the gap. Everyone piles in → futures falls.
- **Futures below spot in the delivery period?** Go long the future, take
  delivery, get the asset cheap. Buyers pile in → futures rises.

So the gap — the **basis** — gets crushed to roughly zero right at delivery.
Before then the future can trade *above* spot (storage + funding costs) or
*below* it (a convenience yield — people who need the physical **now** will pay
up for it). Which regime you're in is a later chapter's headache; the convergence
is not optional.

In [ ]:
T = 60
t = np.arange(T + 1)
spot = 100 + np.cumsum(rng.normal(0, 0.4, T + 1)); spot -= spot[0] - 100
decay = 1 - t / T                       # 1 at the start, 0 at delivery

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, b0, tag, txt in [
    (axes[0], +6.0, "(a)", "futures ABOVE spot\n(carry: storage + funding)"),
    (axes[1], -6.0, "(b)", "futures BELOW spot\n(convenience yield: want it NOW)"),
]:
    fut = spot + b0 * decay
    ax.plot(t, spot, lw=2, label="Spot price")
    ax.plot(t, fut,  lw=2, label="Futures price")
    ax.axvline(T, color="gray", ls="--", lw=.8)
    ax.annotate("delivery:\nbasis → 0", (T, spot[-1]),
                (T - 22, spot[-1] + (5 if b0 > 0 else -7)),
                arrowprops=dict(arrowstyle="->", color="gray"))
    ax.set_title(f"{tag}  {txt}")
    ax.set_xlabel("Time  (last tick = delivery)")
    ax.legend(loc="upper center", fontsize=8)
axes[0].set_ylabel("Price")
fig.suptitle("Whatever the basis is early on, it gets crushed to zero at delivery", y=1.03)
fig.tight_layout(); plt.show()

## 5 · Margin accounts: the exchange politely holds your wallet

Two strangers promising to trade later is a broken promise waiting to happen. The
exchange fixes it with a **margin account**:

- **Initial margin** — cash you post up front (gold: ~$6,000 per contract).
- **Marking to market / daily settlement** — every day at the close, your day's
  gain or loss is moved *into or out of* the account. Winners are literally paid
  the losers' cash that night; that daily flow is **variation margin**.
- **Maintenance margin** (~75% of initial) — the trip-wire. Fall below it and you
  get a **margin call**: top the account back up to the *initial* level by
  tomorrow, or the broker closes you out.
- Interest is usually paid on the balance, so it isn't a real cost. Treasury
  bills can substitute for cash at ~90% of value — a **haircut**.
- **Symmetry**: shorting a future costs the same margin as going long. The spot
  market isn't like that — shorting there means borrowing the asset first.

The classic worked example: you go **long 2 gold contracts** (100 oz each → 200
oz) at **$1,750**. Initial margin **$12,000**, maintenance **$9,000**. Watch what
16 days of ordinary wiggling does to the account.

In [ ]:
settle = [1741.00, 1738.30, 1744.60, 1741.30, 1740.10, 1736.20, 1729.90,
          1730.80, 1725.40, 1728.10, 1711.00, 1711.00, 1714.30, 1716.10,
          1723.00, 1726.90]
entry, oz, initial, maint = 1750.00, 200, 12_000, 9_000

bal, prev, rows, balances = initial, entry, [], [float(initial)]
for day, s in enumerate(settle, start=1):
    gain = oz * (s - prev)
    bal += gain
    call = 0.0
    if bal < maint:                    # topped straight back to the initial level
        call, bal = initial - bal, initial
    rows.append((day, s, gain, bal, call))
    balances.append(bal)
    prev = s

rule("Margin account — long 2 gold futures @ $1,750  (initial $12,000 / maint $9,000)")
print(f"{'Day':>3} {'Settle':>9} {'Day gain':>10} {'Balance':>10} {'Margin call':>12}")
for day, s, gain, b, call in rows:
    tag = "—" if call == 0 else f"{call:,.0f}"
    print(f"{day:>3} {s:>9,.2f} {gain:>10,.0f} {b:>10,.0f} {tag:>12}")
print(f"\nIn @ ${entry:,.2f} on day 1, out @ ${settle[-1]:,.2f} on day 16"
      f"  →  cumulative P&L ${oz*(settle[-1]-entry):,.0f}")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(range(0, 17), balances, marker="o", lw=1.8, label="Margin balance")
ax.axhline(initial, color="tab:green", ls="--", lw=1, label="Initial margin  $12,000")
ax.axhline(maint,   color="tab:red",   ls="--", lw=1, label="Maintenance  $9,000")
for day, s, gain, b, call in rows:
    if call:
        ax.annotate(f"margin call\n+${call:,.0f}", (day, maint), (day, maint - 2700),
                    ha="center", color="tab:red",
                    arrowprops=dict(arrowstyle="->", color="tab:red"))
ax.set_xlabel("Trading day"); ax.set_ylabel("Balance ($)")
ax.set_title("Two dips below the red line, two phone calls from your broker")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

The account is topped back to $12,000 after each call, so on the good days
afterward it sits **above** the initial line — that excess is yours to withdraw
(here we leave it). A futures position is, in effect, **closed out and rewritten
at a new price every single day**.

Now the fun question: **how likely is a margin call in the first place, and how
much worse does it get if volatility ticks up?** Simulate 40,000 price paths.

In [ ]:
def margin_stats(vol_annual, n_paths=40_000, n_days=20, seed=7):
    r = np.random.default_rng(seed)
    F, dt, oz, initial, maint = 1750.0, 1/252, 200, 12_000, 9_000
    shocks = r.normal(0, vol_annual * F * np.sqrt(dt), size=(n_paths, n_days))
    prices = F + np.cumsum(shocks, axis=1)
    bal = np.full(n_paths, float(initial))
    prev = np.full(n_paths, F)
    ever_called = np.zeros(n_paths, bool)
    total_vm = np.zeros(n_paths)
    for k in range(n_days):
        gain = oz * (prices[:, k] - prev)
        bal += gain
        total_vm += np.abs(gain)
        hit = bal < maint
        ever_called |= hit
        bal[hit] = initial
        prev = prices[:, k]
    return ever_called.mean(), total_vm.mean()

vols = [0.10, 0.15, 0.20, 0.30, 0.40]
probs, vms = zip(*(margin_stats(v) for v in vols))
labels = [f"{int(v*100)}%" for v in vols]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.bar(labels, np.array(probs) * 100, color="tab:red", alpha=.8)
a1.set_title("P(at least one margin call in 20 days)"); a1.set_ylabel("%")
a1.set_xlabel("gold volatility (annualised)")
a2.bar(labels, vms, color="tab:blue", alpha=.8)
a2.set_title("Average total variation margin shuffled"); a2.set_ylabel("$")
a2.set_xlabel("gold volatility (annualised)")
fig.suptitle("More volatility → more phone calls and more cash sloshing around", y=1.03)
fig.tight_layout(); plt.show()

for v, p, m in zip(vols, probs, vms):
    print(f"  vol {int(v*100):>2}%   P(call) {p:6.1%}   avg variation margin ${m:,.0f}")

## 5½ · Optional deep cut — your margin account is *literally* a barrier option

A maintenance-margin breach is a **down-and-in barrier** on your futures P&L,
**monitored daily** — which is exactly how a daily-settled future works. So
"how much margin pain am I in for, and how fast does it grow if volatility rises?"
is a **barrier-option price plus its Greeks**.

With adjoint automatic differentiation you **record the payoff once** and get the
premium *and* its sensitivity to price, volatility and time from **one reverse
sweep** — no bump-and-revalue, no re-running the simulation per input.

The next cell is self-contained: **on Google Colab its first run builds the
project** (JDK 25 + Maven + checkout, a few minutes, once per runtime — pick a
*T4 GPU* runtime for the `cuda` engine); **locally** it uses whatever
`nablatensor` you already have. If the bridge can't be made available it prints a
note and moves on — the pure-Python Monte Carlo above already made the
qualitative point.

Setup: futures at **F₀ = 1750**, maintenance-implied trip level
**L = 1750 − (12,000 − 9,000) / 200 = 1735**, strike set to L (so the payoff
measures how far *past* the buffer you blow), horizon 20 trading days, one
monitoring step per daily settlement.

In [ ]:
# --- Optional adjoint section. Runs anywhere; only the Colab branch does work. ---
# Colab: the first run installs JDK 25 + Maven, clones the repo, builds
#        */target/classes and pip-installs the bridge (a few minutes, once per
#        runtime). Local: this just picks up whatever nablatensor you already have
#        (env NABLATENSOR_HOME, else the bridge finds its own checkout). Either
#        way, if the bridge can't be made available the section skips cleanly.
import os, sys, subprocess

ON_COLAB = "google.colab" in sys.modules

if not ON_COLAB:
    PROJECT_ROOT = os.environ.get("NABLATENSOR_HOME")   # None → bridge finds its own checkout
else:
    PROJECT_ROOT = "/content/nablatensor"
    JDK_HOME = "/opt/jdk-25"
    if not os.path.isdir(PROJECT_ROOT + "/.git"):
        _script = r"""
            set -eux
            # JDK 25 — the project's LTS baseline. Colab's apt has no openjdk-25, so
            # take the Temurin GA build straight from the Adoptium API.
            if [ ! -x "$JDK_HOME/bin/java" ]; then
                curl -fsSL -o /tmp/jdk25.tgz \
                  "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse"
                mkdir -p "$JDK_HOME"
                tar -xzf /tmp/jdk25.tgz -C "$JDK_HOME" --strip-components=1
            fi
            command -v mvn >/dev/null 2>&1 || { apt-get -qq update && apt-get -qq install -y maven; }
            [ -d "$PROJECT_ROOT/.git" ] || \
              git clone --depth 1 https://github.com/nablatensor-dev/nablatensor.git "$PROJECT_ROOT"
            cd "$PROJECT_ROOT"
            JAVA_HOME="$JDK_HOME" MAVEN_OPTS=--sun-misc-unsafe-memory-access=allow \
              mvn -q -T1C install -Dmaven.test.skip=true
            pip -q install ./python
        """
        subprocess.run(["bash", "-c", _script], check=True,
                       env={**os.environ, "JDK_HOME": JDK_HOME, "PROJECT_ROOT": PROJECT_ROOT})
    os.environ["JAVA_HOME"] = JDK_HOME
    os.environ["PATH"] = JDK_HOME + "/bin:" + os.environ["PATH"]
    # A Colab GPU runtime keeps the CUDA driver libs here; harmless without a GPU.
    os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

try:
    import nablatensor as nt
    nt.start(project_root=PROJECT_ROOT)
    HAVE_NT = True
except Exception as exc:                                        # noqa: BLE001
    HAVE_NT = False
    print("nablatensor not available here — skipping the adjoint demo.\n  reason:", exc)

if HAVE_NT:
    avail = {e["name"] for e in nt.engines() if e["available"]}
    ENGINE = next((x for x in ("cuda", "vulkan", "rocm", "cpu-jit") if x in avail), "cpu-jit")
    print("adjoint engine:", ENGINE)

    F0, vol, horizon = 1750.0, 0.20, 20
    L = 1735.0
    mkt = nt.EquityMarket(F0, L, vol, 0.0, horizon / 252)       # strike = barrier = L, rate 0
    margin_risk = nt.ExoticProducts.barrier(
        nt.OptionType.PUT, nt.ExoticProducts.Barrier.DOWN_IN, L, 0.005 * F0)
    print("recording payoff:", margin_risk)

    mc = (nt.MonteCarlo.of(margin_risk).market(mkt)
            .steps(horizon).fp32().greeks().on(ENGINE).build())
    res = mc.run(4_000_000, 42)
    g = res.greeks()
    rule("Margin-risk 'premium' and its sensitivities — ONE reverse sweep")
    print(f"  premium  (expected breach depth)   {res.price():10.4f}")
    print(f"  d/dF0    (drifting toward the wire) {g.spot():10.4f}")
    print(f"  d/dVol   (vol ↑  ⇒  margin pain ↑)  {g.vol():10.4f}   <-- the whole point")
    print(f"  d/dT     (more days, more risk)     {g.maturity():10.4f}")
    print(f"  ({res.scenarios():,} paths, {mc.nodes()} tape nodes, engine {mc.engine()})")

    grid = np.arange(1700, 1801, 5.0)
    prem, dlt = [], []
    for s in grid:
        r = mc.run(mkt.withSpot(float(s)), 800_000, 42)
        prem.append(r.price()); dlt.append(r.greeks().spot())
    mc.close()

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(grid, prem, marker="o"); a1.axvline(L, color="tab:red", ls="--", lw=1)
    a1.set_title("Margin-risk premium vs futures price")
    a1.set_xlabel("F₀"); a1.set_ylabel("premium")
    a2.plot(grid, dlt, marker="o", color="tab:purple")
    a2.axvline(L, color="tab:red", ls="--", lw=1)
    a2.set_title("…and its delta (one reverse sweep per point)")
    a2.set_xlabel("F₀")
    fig.suptitle("Near the red trip-wire the risk goes non-linear — exactly where bumping gets noisy",
                 y=1.03)
    fig.tight_layout(); plt.show()
else:
    print("(No adjoint numbers this run — the Monte Carlo above already showed: vol up ⇒ calls up.)")

## 6 · The clearing house: the middleman nobody minds

The **clearing house** stands between every buyer and seller and guarantees both.
Its **members** post initial margin (a.k.a. *clearing margin*) for all contracts
they clear — here maintenance = initial — and settle daily. If a member's book
lost money overall, it wires **variation margin** to the clearing house; if it
gained, it receives variation margin. On a wild day the clearing house can demand
**intraday** variation margin too.

Margins are figured **net**, not gross: a member carrying one client **long 20**
and another **short 15** posts margin on just **5** contracts. The calculation
aims for roughly **99% confidence** that a defaulting member's margin covers the
close-out losses. Members also pay into a **guaranty fund** for when a member
blows clean through its margin.

In [ ]:
rule("Net vs gross margining")
long_c, short_c, per = 20, 15, 2_000
print(f"  Client A long {long_c}, client B short {short_c}")
print(f"  Gross:  margin on {long_c + short_c} contracts = ${(long_c + short_c) * per:,}")
print(f"  Net:    margin on {abs(long_c - short_c)} contracts = ${abs(long_c - short_c) * per:,}")

rule("Clearing-house top-up — a day in the life")
# start: long 100 @ settle 50,000, original margin $2,000/contract
# next day: clears 20 MORE longs entered @ 51,000; that day's settle = 50,200
existing, s0, om = 100, 50_000, 2_000
new, new_px, s1 = 20, 51_000, 50_200
variation = existing * (s1 - s0) + new * (s1 - new_px)
extra_initial = new * om
print(f"  Variation margin: 100×(50,200−50,000) + 20×(50,200−51,000) = ${variation:,}")
print(f"  Extra initial margin for {new} new contracts:                 ${extra_initial:,}")
print(f"  Net to ADD to the margin account:                             ${extra_initial - variation:,}")

## 7 · Credit risk, and the day it got stress-tested (19 October 1987)

The whole margin machine exists so that **winners actually get paid**. It has
worked: on the major exchanges, futures contracts have always been honoured.

The scare: **19 October 1987**, the S&P 500 fell more than **20% in a day**.
Traders long S&P 500 futures went *negative* in their margin accounts. Some
didn't meet the call and were closed out still owing money; some never paid; a
few brokers went bankrupt because, without their clients' cash, they couldn't
meet their own calls. But the clearing houses had enough to make sure **every
single short position got paid**. The system bent; it didn't break.

## 8 · OTC markets: futures' scruffy, customisable cousin

**Over-the-counter** = two firms strike a derivative directly, no exchange.
Flexible, but each side carries the other's default risk. Two ways to tame it:

- **Central counterparty (CCP)** — a clearing house for *standardised* OTC
  trades. Present a trade; the CCP becomes the counterparty to **both** sides,
  takes both credit risks, and demands initial margin, daily variation margin and
  guaranty-fund contributions. Since the 2007–09 crisis, regulators **require**
  most standard OTC trades between financial institutions to be cleared this way.
- **Bilateral clearing** — everything else. A and B sign a master agreement
  (usually an ISDA) with a **Credit Support Annex (CSA)**: mark to market daily,
  and whoever is underwater posts **collateral** (variation margin by another
  name). Initial margin used to be rare here; since 2016, big financials must
  post both. Securities used as collateral get a **haircut** off their market value.

In [ ]:
n = 8
ang = np.linspace(0, 2*np.pi, n, endpoint=False)
xy = np.c_[np.cos(ang), np.sin(ang)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.4))
for ax in (a1, a2):
    ax.set_aspect("equal"); ax.axis("off")

pairs = 0
for i in range(n):
    for j in range(i + 1, n):
        a1.plot(*zip(xy[i], xy[j]), color="tab:red", alpha=.40, lw=1); pairs += 1
a1.scatter(*xy.T, s=520, zorder=5, color="white", edgecolor="black")
for k, (x, y) in enumerate(xy):
    a1.text(x, y, chr(65 + k), ha="center", va="center", weight="bold")
a1.set_title(f"(a) Bilateral: {pairs} separate agreements\nevery dealer wired to every other")

for i in range(n):
    a2.plot([xy[i, 0], 0], [xy[i, 1], 0], color="tab:blue", alpha=.55, lw=1.3)
a2.scatter(*xy.T, s=520, zorder=5, color="white", edgecolor="black")
for k, (x, y) in enumerate(xy):
    a2.text(x, y, chr(65 + k), ha="center", va="center", weight="bold")
a2.scatter([0], [0], s=1500, color="gold", edgecolor="black", zorder=6)
a2.text(0, 0, "CCP", ha="center", va="center", weight="bold")
a2.set_title(f"(b) Central: {n} links to one CCP\nit becomes everyone's counterparty")
fig.tight_layout(); plt.show()

print("  dealers   bilateral wires   with one CCP")
for m in (8, 20, 50, 100):
    print(f"    {m:>3}          {m*(m-1)//2:>6}            {m:>4}")
print("\n  Bilateral wiring grows like n² ; the hub grows like n.")

> **Field note — the hedge fund that did arbitrage until it hit a wall.**
> A 1990s hedge fund ran *convergence arbitrage*: find two near-identical bonds
> from the same issuer where the **less-traded** one trades cheap purely because
> nobody wants the illiquidity. **Buy the cheap one, short the rich one, wait**
> for the prices to converge, collect the difference. Every trade was
> collateralised bilaterally, which let them lever up *enormously* — the
> collateral flows on the two legs roughly cancel, so the leverage felt free.
> Then **August 1998**: Russia defaults, everyone stampedes into liquid assets,
> the cheap bond gets *cheaper* and the rich bond gets *richer* — the spread
> **diverged**. They owed collateral on both legs at once, were levered roughly
> 25-to-1, and couldn't pay. Positions were force-closed; about **$4 billion**
> evaporated. The trade was probably right *eventually* — but "eventually"
> requires still being solvent.

## 9 · Futures vs OTC: the small cash-flow differences

- Initial margin posted as cash usually **earns interest** in both worlds.
- **Futures variation margin earns no interest** — it *is* the daily settlement.
  It's a payment, not a deposit.
- **OTC variation margin (CCP or CSA) does earn interest** — those trades aren't
  settled daily, so the cash you posted is still yours, just parked with the
  other side.

## 10 · Reading a futures quote board

Each row of a quote table — say June 2020 gold, 100 oz, dollars per ounce:

- **Open / High / Low** — first trade of the day and the day's range so far.
- **Prior settlement** — yesterday's official mark; daily P&L and margin were
  figured from it.
- **Last** — the most recent trade. **Change** = Last − prior settlement.
- **Settlement price** — *today's* official mark, roughly the price right before
  the close. If June gold's Last of $1,725.5 becomes today's settlement, a
  one-contract **long loses** (1,752.1 − 1,725.5) × 100 = **$2,660** today and
  the short gains it.
- **Volume** — contracts traded today. **Open interest** — contracts *still
  outstanding* (number of longs = number of shorts). Heavy day-trading can push
  volume *above* open interest.
- **Normal market**: futures price **rises** with maturity. **Inverted market**:
  it **falls** with maturity. On one May-2020 day, gold, crude, corn and wheat
  looked normal; soybeans and live cattle were a mix of both.

In [ ]:
gold = [   # month,     open,   high,   low,   prior,  last,   change, volume
    ("Jun 2020", 1751.7, 1751.7, 1715.3, 1752.1, 1725.5, -26.6, 223_200),
    ("Aug 2020", 1765.3, 1765.3, 1731.2, 1765.6, 1740.7, -24.9,  54_503),
    ("Oct 2020", 1768.0, 1768.8, 1739.0, 1774.0, 1747.4, -26.6,   2_559),
    ("Dec 2020", 1778.8, 1779.8, 1743.8, 1781.7, 1752.7, -29.0,   5_280),
    ("Dec 2021", 1779.0, 1779.0, 1755.1, 1790.7, 1757.2, -33.5,     345),
]
rule("Gold futures — 100 oz, $/oz  (one day in May 2020)")
print(f"{'Contract':>9} {'Open':>8} {'High':>8} {'Low':>8} {'Prior':>8} {'Last':>8} {'Chg':>7} {'Volume':>9}")
for m, o, h, l, p, la, c, v in gold:
    print(f"{m:>9} {o:>8.1f} {h:>8.1f} {l:>8.1f} {p:>8.1f} {la:>8.1f} {c:>7.1f} {v:>9,}")

jun = gold[0]
print(f"\nIf June's last ${jun[5]:.1f} is the settlement, a 1-lot long loses "
      f"${(jun[4]-jun[5])*100:,.0f} today; the short gains it.")

months = [g[0] for g in gold]
prior  = np.array([g[4] for g in gold])
inverted = prior[0] - (prior - prior[0]) * 1.6           # a made-up mirror image

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, prior, marker="o", lw=2,
        label="Gold (prior settle) — NORMAL: rises with maturity")
ax.plot(months, inverted, marker="s", ls="--", lw=2,
        label="Made-up INVERTED market — falls with maturity")
ax.set_ylabel("Futures price")
ax.set_title("Normal vs inverted: which way does the curve lean?")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

## 11 · Delivery: for the stubborn 2%

The delivery window is set by the exchange. The **short** decides *when* to
deliver and files a **notice of intention to deliver** (how many contracts,
where, and which grade). The exchange passes that notice to a **long** — usually
the **oldest outstanding long position** — who **must** accept it (a transferable
notice can sometimes be handed to another willing long).

- **Commodity**: you accept a warehouse receipt, pay, and the storage bills are
  now yours (livestock: also the feeding — see the cattle story).
- **Financial**: usually a wire transfer.
- Three key dates: **first notice day**, **last notice day**, **last trading
  day** (a few days before last notice). Long and don't want a surprise herd?
  **Close out before first notice day.**
- **Cash settlement**: for things you can't hand over (the S&P 500 is 500
  separate stocks), every contract is simply closed on a set day at the spot
  level — S&P 500 futures use the third-Friday **opening** price.

## 12 · The trader zoo and the order menu

**Who's trading**

- **FCMs** (futures commission merchants) execute for clients and charge commission.
- **Locals** trade their own money.
- Speculators come in three speeds: **scalpers** (seconds to minutes, tiny
  moves), **day traders** (flat by the close — no overnight news risk), **position
  traders** (weeks to months, chasing the big move).

**How you tell the broker what to do**

- **Market** — fill me now at whatever's available.
- **Limit** — fill only at my price or better (may never fill).
- **Stop / stop-loss** — becomes a *market* order once the price trades to my
  trigger; used to cap a loss.
- **Stop-limit** — becomes a *limit* order at the trigger (needs two prices, stop
  and limit).
- **Market-if-touched (MIT)** — becomes a *market* order once the price touches my
  level; used to *take profit*.
- **Discretionary / market-not-held** — the broker may wait for a better fill.
- **Time in force** — day order (the default), **good-till-cancelled**,
  **fill-or-kill** (now or never), **time-of-day**.

In [ ]:
n_days = 40
path = 100 + np.cumsum(rng.normal(0.05, 1.0, n_days))
limit_buy, stop_sell, mit_sell = 97.0, 96.0, 104.0

def first_cross(p, level, going_up):
    idx = np.where(p >= level if going_up else p <= level)[0]
    return int(idx[0]) if len(idx) else None

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(path, lw=1.8, color="0.4")
for lvl, idx, txt, col, up in [
    (limit_buy, first_cross(path, limit_buy, False), "LIMIT BUY fills\n(price or better)", "tab:green", False),
    (stop_sell, first_cross(path, stop_sell, False), "STOP-SELL triggers\n(cap the loss)", "tab:red", False),
    (mit_sell,  first_cross(path, mit_sell, True),  "MIT SELL triggers\n(take the profit)", "tab:blue", True),
]:
    ax.axhline(lvl, color=col, ls="--", lw=1)
    if idx is not None:
        ax.scatter([idx], [path[idx]], color=col, zorder=5, s=60)
        ax.annotate(txt, (idx, lvl), (idx + 0.6, lvl + (3 if up else -4)),
                    color=col, fontsize=8)
ax.set_xlabel("Trading day"); ax.set_ylabel("Price")
ax.set_title("Same path, three orders, three different trigger moments")
fig.tight_layout(); plt.show()

## 13 · Referees: who regulates this circus

- **CFTC** (US, since 1974) — makes sure prices are public, big positions are
  reported, brokers are licensed and background-checked, complaints get handled;
  can force exchanges to discipline members.
- **NFA** (industry self-regulator, since 1982) — monitors trading, disciplines,
  arbitrates disputes.
- **Dodd–Frank** (2010) — expanded the CFTC: standard OTC swaps between financial
  institutions must trade on swap-execution facilities and clear through CCPs.

**Cornering the market**: take a giant long position *and* quietly control the
physical supply; as maturity nears, refuse to close out; the shorts can't find
enough physical to deliver and panic; both futures **and** spot spike.
Regulators respond by hiking margins, tightening position limits, or forcing
liquidation. The legendary case: two brothers and **silver, 1979–80 — about
$6/oz to nearly $50/oz**, then straight back down.

In [ ]:
mo = np.arange(0, 13)          # mid-1979 → early 1980, monthly
silver = np.array([6, 6.5, 7.5, 9, 11, 15, 20, 28, 35, 44, 50, 39, 11.0])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mo, silver, marker="o", lw=2, color="tab:gray")
ax.annotate("the corner peaks near $50", (10, 50), (4.5, 45),
            arrowprops=dict(arrowstyle="->"))
ax.annotate("margin hikes + forced\nliquidation → collapse", (12, 11), (7, 20),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("months (mid-1979 → 1980)"); ax.set_ylabel("silver, $/oz")
ax.set_title("What cornering a market looks like — on the way up, then down (stylised)")
fig.tight_layout(); plt.show()

## 14 · Accounting & tax, the two-minute version

- **Hedge accounting** — if the futures position *qualifies* as a hedge, its
  gains and losses are recognised in the **same period** as the thing being
  hedged (timing matched). Otherwise you mark it to market as it moves.
  (US: FAS 133 / ASC 815. International: IAS 39 / IFRS 9.)
- **Worked example**: buy one March-2021 corn contract (5,000 bu) in September
  2020 at **350 ¢/bu**; it's **370** at year-end 2020 and **380** when you close
  in February 2021.
  - *Not a hedge*: **+$1,000 booked in 2020** (5,000 × (3.70 − 3.50)), **+$500 in
    2021** (5,000 × (3.80 − 3.70)).
  - *Qualifies as a hedge* (you're hedging a February-2021 corn purchase): the
    whole **+$1,500 lands in 2021**, matching the purchase.
- **Tax (US, non-corporate)** — futures are marked to market at year-end even if
  still open, and the gain/loss is treated **60% long-term / 40% short-term**
  regardless of how long you held it (the **"60/40 rule"**). Bona-fide hedging
  transactions are exempt and get ordinary-income treatment, timing matched.

In [ ]:
years = ["2020", "2021"]
not_hedge = [1000, 500]
is_hedge  = [0, 1500]
x = np.arange(2); w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, not_hedge, w, label="Not a hedge — recognise as it moves")
ax.bar(x + w/2, is_hedge,  w, label="Qualifies as a hedge — match the purchase")
ax.set_xticks(x, years); ax.set_ylabel("gain recognised ($)")
ax.set_title("Same $1,500 profit — hedge accounting only changes *when* you book it")
ax.legend(); fig.tight_layout(); plt.show()

## 15 · Forward vs Futures: same idea, different plumbing

| Forward | Futures |
|---|---|
| Private contract between two parties | Traded on an exchange |
| Not standardised | Standardised contract |
| Usually one delivery date | A range of delivery dates |
| Settled at the end of its life | Settled every day |
| Some credit risk | Virtually no credit risk |

**Profit timing** — sterling, both priced at **1.2000** for delivery in 90 days,
spot ends at **1.4000**:

- **Forward**, long £1,000,000 → the whole **+$200,000** shows up on **day 90**.
- **Futures**, long 16 contracts (£62,500 each = £1,000,000) → the same
  **+$200,000**, but dribbled out day by day: some up days, some down days, same
  total.

**FX quoting quirk** — futures always quote **USD per unit of the foreign
currency**. Forwards quote like spot: for GBP, EUR, AUD, NZD that's also USD per
unit (directly comparable to futures), but for other currencies it's *units per
USD*. A Canadian-dollar futures quote of **0.7500 USD/CAD** is the same price as
a forward quote of **1.3333 CAD/USD**.

In [ ]:
days = np.arange(0, 91)
S0, ST, notional = 1.2000, 1.4000, 1_000_000

fx = np.concatenate([[S0], S0 + np.cumsum(rng.normal(0, 0.004, 90))])
fx += np.linspace(0, ST - fx[-1], 91)          # pin the endpoint to exactly 1.4000

futures_cum = notional * (fx - S0)
forward_cum = np.zeros(91); forward_cum[-1] = notional * (ST - S0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, futures_cum, lw=1.7, label="Futures: marked daily (jagged)")
ax.plot(days, forward_cum, lw=2, ls="--", label="Forward: nothing… then a cliff on day 90")
ax.axhline(notional * (ST - S0), color="gray", lw=.8)
ax.set_xlabel("Day"); ax.set_ylabel("Cumulative P&L ($)")
ax.set_title("Forward vs futures: identical destination, very different ride")
ax.legend(); fig.tight_layout(); plt.show()

print(f"Both land at ${notional*(ST-S0):,.0f}.   CAD check: 1 / 0.7500 = {1/0.75:.4f} CAD per USD")

## The cheat sheet (screenshot this one)

- **Long / short** — agreed to buy / agreed to sell.
- **Closing out** — do the opposite trade before delivery; P&L = futures-price change.
- **Contract size / grade / delivery month** — all fixed by the exchange; that's the point of a *future*.
- **Basis** = futures − spot. **Convergence** — basis → 0 at delivery, enforced by arbitrage.
- **Initial margin** — posted up front. **Maintenance margin** — the trip-wire (~75% of initial).
- **Margin call** — top back up to *initial* by tomorrow, or get closed out.
- **Variation margin / marking to market** — daily cash flow of gains and losses.
- **Clearing house** — guarantees both sides; margins figured **net**, not gross; **guaranty fund** behind it.
- **CCP** — clearing house for standard OTC trades; becomes counterparty to both sides.
- **Bilateral clearing / CSA / collateral / haircut** — the OTC do-it-yourself version.
- **Settlement price** — the day's official mark. **Volume** vs **open interest** — traded today vs still outstanding.
- **Normal vs inverted market** — futures price rises vs falls with maturity.
- **First / last notice day, last trading day** — the delivery calendar; close out before first notice day.
- **Cash settlement** — for underliers you can't deliver (stock indices).
- **FCM / local**; **scalper / day trader / position trader**.
- **Market / limit / stop / stop-limit / MIT / discretionary** orders; **GTC**, **fill-or-kill**.
- **CFTC / NFA / Dodd–Frank** — the referees. **Cornering** — squeeze the shorts by controlling supply.
- **Hedge accounting** — match the timing. **60/40 rule** — futures tax split, any holding period.

## Pop quiz

Have a go, then run the next cell for the answers.

1. You're **short** July silver at $17.20 (5,000 oz), initial margin $4,000,
   maintenance $3,000. What price move triggers a margin call? What if you ignore it?
2. The futures price of a commodity is **above** the spot price *during the
   delivery period*. Is there an arbitrage? What if it's below?
3. A clearing-house member is **long 100** contracts at settle $50,000 (original
   margin $2,000/contract). Next day it also clears **20 more longs** entered at
   $51,000; that day's settle is $50,200. How much must it add to its account?
4. What is a **stop order to sell at $2**, and when would you use it? How is a
   **limit order to sell at $2** different?
5. A long forward to buy at *K* **plus** a long put struck at *K*, same date —
   what single position is that equal to?

In [ ]:
answers = [
 ("Short July silver, init $4,000 / maint $3,000 — what triggers a call?",
  "A $1,000 loss does it: 1,000 / 5,000 = $0.20/oz. You're SHORT, so a rise to "
  "$17.40 triggers the call. Ignore it and the broker closes the position out."),
 ("Futures above spot during the delivery period — arbitrage?",
  "Yes: short the future, buy the asset, deliver — lock in (F − S) > 0. Everyone "
  "doing it drags F down to S. If F < S during delivery there is no clean "
  "arbitrage for most commodities (you'd have to short the physical, usually "
  "impossible), so F can sit a little below S."),
 ("Clearing-house member: long 100 @ 50,000, then +20 longs @ 51,000, settle 50,200 — add how much?",
  "Variation margin = 100×(50,200−50,000) + 20×(50,200−51,000) = +20,000 − 16,000 "
  "= +$4,000. Extra initial margin for 20 contracts = $40,000. Add "
  "40,000 − 4,000 = $36,000."),
 ("Stop order to sell at $2 vs limit order to sell at $2.",
  "STOP: dormant until the price trades down to $2, then becomes a market sell — "
  "used to cap a loss / protect a profit on a long. LIMIT: fills only at $2 or "
  "higher — used to sell into strength."),
 ("Long forward to buy at K  +  long put struck at K, same date.",
  "A long call struck at K.  Forward = S_T − K ; put = max(K − S_T, 0) ; "
  "sum = max(S_T − K, 0)."),
]
for i, (q, a) in enumerate(answers, 1):
    print(f"\nQ{i}. {q}\n  ➜ {a}")

---

You now understand the plumbing of futures markets better than plenty of people
who trade them. Go forth — and try not to accidentally buy a herd of cattle.